# Virginia Piedmont native phenology

This notebook analyzes seasonal prevalence of native Lepidoptera and Anthophila in the Virginia Piedmont using iNaturalist's aggregate histogram endpoints rather than paginated raw observation fetches.

**Approach:**
- `/observations/histogram` (week_of_year and month_of_year) returns complete all-time counts in a single API call per query — no pagination, no truncation.
- `native=true` delegates nativity filtering to iNat's own establishment-means index; for county-level place IDs, iNat resolves nativity up the ancestor hierarchy to Virginia state level.
- Effort normalization divides each group's weekly count by total verifiable observations in the same week, correcting for observer seasonality.
- Per-taxon detail uses `species_counts` (also unpaginated) to identify top native taxa, then one monthly histogram call per taxon.

**Total API calls: ~24** (vs ~175+ for the prior paginated approach, which captured ≤3% of Lepidoptera observations).

**Outputs:**
- Weekly overview chart: raw counts + effort-normalized fraction, all focus groups and life stage slices
- Monthly outreach chart: top-N native taxa per group, bar chart suitable for poster/web use
- Annotation coverage report: flags sparse life stage slices

In [8]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
from IPython.display import display
from ipywidgets import Dropdown, SelectMultiple

repo_root = Path.cwd()
if not (repo_root / 'helpers.py').exists() and (repo_root.parent.parent / 'helpers.py').exists():
    repo_root = repo_root.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from helpers import get_inat_session

# 27 Piedmont counties + 9 independent cities (iNaturalist place IDs)
PIEDMONT_PLACE_IDS = [
    # Counties
    2913,   # Albemarle
    1660,   # Amelia
    1372,   # Appomattox
    1719,   # Arlington
    733,    # Brunswick
    2589,   # Buckingham
    617,    # Campbell
    1662,   # Charlotte
    2917,   # Culpeper
    2590,   # Cumberland
    738,    # Fairfax
    1494,   # Fauquier
    1721,   # Fluvanna
    2920,   # Goochland
    1075,   # Halifax
    1186,   # Henry
    3032,   # Louisa
    739,    # Loudoun
    1724,   # Lunenburg
    1493,   # Mecklenburg
    742,    # Nottoway
    743,    # Orange
    1664,   # Pittsylvania
    1491,   # Powhatan
    2923,   # Prince Edward
    744,    # Prince William
    2925,   # Spotsylvania
    # Independent cities
    13446,  # Alexandria
    13435,  # Charlottesville
    13434,  # Danville
    108238, # Fairfax (city)
    13451,  # Falls Church
    13483,  # Lynchburg
    13410,  # Manassas
    13430,  # Manassas Park
    13445,  # Martinsville
]

TOP_N = 25
MIN_OBS_PER_YEAR = 25
MAX_COMPARE_TAXA = 3
USE_COMPARE_MODE = False
REFRESH_CACHE = False

TAXON_REGISTRY = [
    {
        'key': 'lepidoptera',
        'focus_group': 'Lepidoptera',
        'title_label': 'Butterflies and Moths',
        'taxon_id': 47157,
        'extra': {},
        'color': '#1f77b4',
        'life_stages': [
            {'bucket': 'adult', 'term_id': 1, 'term_value_id': 2},
            {'bucket': 'larva', 'term_id': 1, 'term_value_id': 6},
            {'bucket': 'pupa', 'term_id': 1, 'term_value_id': 4},
        ],
    },
    {
        'key': 'anthophila',
        'focus_group': 'Anthophila',
        'title_label': 'Bees',
        'taxon_id': 630955,
        'extra': {},
        'color': '#ff7f0e',
        'life_stages': [],
    },
    {
        'key': 'mammalia',
        'focus_group': 'Mammalia',
        'title_label': 'Mammals',
        'taxon_id': 40151,
        'extra': {},
        'color': '#2ca02c',
        'life_stages': [],
    },
    {
        'key': 'aves',
        'focus_group': 'Aves',
        'title_label': 'Birds',
        'taxon_id': 3,
        'extra': {},
        'color': '#d62728',
        'life_stages': [],
    },
    {
        'key': 'plants_in_flower',
        'focus_group': 'Plants in Flower',
        'title_label': 'Flowering Plants',
        'taxon_id': 47126,
        'extra': {'term_id': 12, 'term_value_id': 13},
        'color': '#9467bd',
        'life_stages': [],
    },
]

_taxon_lookup = {t['key']: t for t in TAXON_REGISTRY}

taxon_selector = Dropdown(
    options=[(t['title_label'], t['key']) for t in TAXON_REGISTRY],
    value='lepidoptera',
    description='Taxon:',
)
compare_taxa_selector = SelectMultiple(
    options=[(t['title_label'], t['key']) for t in TAXON_REGISTRY],
    value=('lepidoptera', 'anthophila'),
    description='Compare:',
)

display(taxon_selector)
display(compare_taxa_selector)


def get_taxon_by_key(key: str) -> dict:
    if key not in _taxon_lookup:
        raise ValueError(f'Unknown taxon key: {key}')
    return _taxon_lookup[key]


def get_active_taxa() -> list[dict]:
    if USE_COMPARE_MODE:
        keys = list(compare_taxa_selector.value)
        if not keys:
            keys = [taxon_selector.value]
        keys = keys[:MAX_COMPARE_TAXA]
    else:
        keys = [taxon_selector.value]
    return [get_taxon_by_key(key) for key in keys]


def apply_plot_style(fig, title_text: str, height: int = 700):
    fig.update_layout(
        title={'text': title_text, 'x': 0.5, 'xanchor': 'center'},
        template='plotly_white',
        height=height,
        hovermode='x unified',
        legend={
            'orientation': 'h',
            'yanchor': 'bottom',
            'y': 1.02,
            'xanchor': 'left',
            'x': 0.0,
            'bgcolor': 'rgba(255,255,255,0.85)',
        },
        margin={'l': 70, 'r': 35, 't': 85, 'b': 60},
        font={'size': 12},
        paper_bgcolor='white',
        plot_bgcolor='rgba(246,248,251,0.65)',
    )
    fig.update_xaxes(showgrid=True, gridcolor='rgba(200,200,200,0.25)', zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(200,200,200,0.25)', zeroline=False)
    return fig


_place_str = ','.join(str(x) for x in PIEDMONT_PLACE_IDS)

print('Taxon options available for this notebook run:')
display(pd.DataFrame([
    {
        'key': t['key'],
        'label': t['title_label'],
        'taxon_id': t['taxon_id'],
        'extra_filters': t['extra'] if t['extra'] else None,
        'life_stage_slices': len(t['life_stages']),
    }
    for t in TAXON_REGISTRY
]))
print(f'Compare mode: {USE_COMPARE_MODE} | Refresh cache: {REFRESH_CACHE}')

HISTOGRAM_URL = 'https://api.inaturalist.org/v1/observations/histogram'
SPECIES_COUNTS_URL = 'https://api.inaturalist.org/v1/observations/species_counts'

CACHE_PATH = repo_root / 'notebooks' / 'exploratory' / 'cache' / 'va_piedmont_phenology'
CACHE_MAX_AGE_DAYS = 7
CACHE_TAG = 'v3_taxon_driven'

session = get_inat_session()

Dropdown(description='Taxon:', options=(('Butterflies and Moths', 'lepidoptera'), ('Bees', 'anthophila'), ('Ma…

SelectMultiple(description='Compare:', index=(0, 1), options=(('Butterflies and Moths', 'lepidoptera'), ('Bees…

Taxon options available for this notebook run:


,key,label,taxon_id,extra_filters,life_stage_slices
0,lepidoptera,Butterflies and Moths,47157,None,3
1,anthophila,Bees,630955,None,0
2,mammalia,Mammals,40151,None,0
3,aves,Birds,3,None,0
4,plants_in_flower,Flowering Plants,47126,"{'term_id': 12, 'term_value_id': 13}",0


Compare mode: False | Refresh cache: False


In [9]:
# -- Preflight: total observation counts for active selection only --
selected_taxa = get_active_taxa()

print('Preflight totals for active selection:')
for taxon in selected_taxa:
    params = {
        'place_id': _place_str,
        'taxon_id': taxon['taxon_id'],
        'verifiable': 'true',
        'per_page': 0,
        **taxon.get('extra', {}),
    }
    resp = session.get('https://api.inaturalist.org/v1/observations', params=params)
    resp.raise_for_status()
    total = int(resp.json().get('total_results', 0))
    print(f"- {taxon['title_label']}: {total:,} verifiable observations")

Preflight totals for active selection:
- Butterflies and Moths: 210,387 verifiable observations


In [10]:
# -- Fetch selected-group histograms and shared effort denominators --
# Cache-first by default. Set REFRESH_CACHE=True in Cell 2 to force a refresh.

selected_taxa = get_active_taxa()
selected_keys = [t['key'] for t in selected_taxa]
selection_signature = '__'.join(sorted(selected_keys))

_base = {'place_id': _place_str, 'verifiable': 'true'}

hist_cache = CACHE_PATH / f'hist_df_{selection_signature}_{CACHE_TAG}.parquet'
effort_cache = CACHE_PATH / f'effort_df_{CACHE_TAG}.parquet'


def _is_fresh(path: Path, max_age_days: int) -> bool:
    if not path.exists():
        return False
    age_days = (pd.Timestamp.now().timestamp() - path.stat().st_mtime) / 86400
    return age_days < max_age_days


need_hist_fetch = REFRESH_CACHE or (not _is_fresh(hist_cache, CACHE_MAX_AGE_DAYS))
need_effort_fetch = REFRESH_CACHE or (not _is_fresh(effort_cache, CACHE_MAX_AGE_DAYS))

if not need_hist_fetch and not need_effort_fetch:
    hist_df = pd.read_parquet(hist_cache)
    effort_df = pd.read_parquet(effort_cache)
    print(f'Loaded histograms from cache: {hist_cache.name}')
    print(f'Loaded effort from cache: {effort_cache.name}')
else:
    api_calls = 0

    if need_hist_fetch:
        hist_rows = []
        query_specs = []
        for taxon in selected_taxa:
            query_specs.append(
                {
                    'focus_group': taxon['focus_group'],
                    'life_stage_bucket': 'overall',
                    'extra': {'taxon_id': taxon['taxon_id'], **taxon.get('extra', {})},
                }
            )
            for stage in taxon.get('life_stages', []):
                query_specs.append(
                    {
                        'focus_group': taxon['focus_group'],
                        'life_stage_bucket': stage['bucket'],
                        'extra': {
                            'taxon_id': taxon['taxon_id'],
                            'term_id': stage['term_id'],
                            'term_value_id': stage['term_value_id'],
                            **taxon.get('extra', {}),
                        },
                    }
                )

        for spec in query_specs:
            for interval in ('week_of_year', 'month_of_year'):
                params = {**_base, **spec['extra'], 'interval': interval, 'native': 'true'}
                response = session.get(HISTOGRAM_URL, params=params)
                response.raise_for_status()
                api_calls += 1
                for key, count in response.json().get('results', {}).get(interval, {}).items():
                    hist_rows.append(
                        {
                            'focus_group': spec['focus_group'],
                            'life_stage_bucket': spec['life_stage_bucket'],
                            'interval': interval,
                            'bin': int(key),
                            'count': int(count),
                        }
                    )
        hist_df = pd.DataFrame(hist_rows)
    else:
        hist_df = pd.read_parquet(hist_cache)

    if need_effort_fetch:
        effort_rows = []
        for interval in ('week_of_year', 'month_of_year', 'year'):
            params = {**_base, 'interval': interval}
            response = session.get(HISTOGRAM_URL, params=params)
            response.raise_for_status()
            api_calls += 1
            for key, count in response.json().get('results', {}).get(interval, {}).items():
                bin_value = pd.to_datetime(key).year if interval == 'year' else int(key)
                effort_rows.append({'interval': interval, 'bin': bin_value, 'effort': int(count)})
        effort_df = pd.DataFrame(effort_rows)
    else:
        effort_df = pd.read_parquet(effort_cache)

    if not hist_df.empty:
        hist_df = hist_df.merge(effort_df, on=['interval', 'bin'], how='left')
        hist_df['fraction'] = hist_df['count'] / hist_df['effort'].replace(0, float('nan'))

    CACHE_PATH.mkdir(parents=True, exist_ok=True)
    hist_df.to_parquet(hist_cache, index=False)
    effort_df.to_parquet(effort_cache, index=False)
    print(f'Fetched with {api_calls} API calls.')
    print(f'Wrote hist cache: {hist_cache.name}')
    print(f'Wrote effort cache: {effort_cache.name}')

print()
print('Verification: weekly native histogram sums by active taxon')
for taxon in selected_taxa:
    fg = taxon['focus_group']
    total = hist_df[
        (hist_df['focus_group'] == fg)
        & (hist_df['life_stage_bucket'] == 'overall')
        & (hist_df['interval'] == 'week_of_year')
    ]['count'].sum()
    print(f"- {taxon['title_label']}: {int(total):,}")

Fetched with 11 API calls.
Wrote hist cache: hist_df_lepidoptera_v3_taxon_driven.parquet
Wrote effort cache: effort_df_v3_taxon_driven.parquet

Verification: weekly native histogram sums by active taxon
- Butterflies and Moths: 164,298


In [ ]:
# -- Verification spot-checks (runs only when Lepidoptera is active) --
selected_taxa = get_active_taxa()
selected_keys = {t['key'] for t in selected_taxa}

if 'lepidoptera' not in selected_keys:
    print('Spot-checks skipped: currently configured only for Lepidoptera runs.')
else:
    spot_checks = [
        {'taxon_id': 48662, 'common_name': 'Monarch', 'expect': 'native', 'peak_months': {4, 5, 9, 10}},
        {'taxon_id': 53827, 'common_name': 'Eastern Tiger Swallowtail', 'expect': 'native', 'peak_months': {5, 6, 8}},
        {'taxon_id': 47219, 'common_name': 'European Honeybee', 'expect': 'absent', 'peak_months': set()},
    ]

    print('Spot-check results:')
    print()
    for sc in spot_checks:
        native_response = session.get(
            SPECIES_COUNTS_URL,
            params={**_base, 'taxon_id': sc['taxon_id'], 'native': 'true', 'per_page': 1},
        )
        native_response.raise_for_status()
        native_count = native_response.json().get('total_results', 0)

        introduced_response = session.get(
            SPECIES_COUNTS_URL,
            params={**_base, 'taxon_id': sc['taxon_id'], 'introduced': 'true', 'per_page': 1},
        )
        introduced_response.raise_for_status()
        introduced_count = introduced_response.json().get('total_results', 0)

        if sc['expect'] == 'absent':
            status = 'OK' if native_count == 0 else f'Unexpected native hits: {native_count}'
            print(
                f"  {sc['common_name']:30s} expect=absent "
                f"native_results={native_count} introduced_results={introduced_count} {status}"
            )
            continue

        histogram_response = session.get(
            HISTOGRAM_URL,
            params={**_base, 'taxon_id': sc['taxon_id'], 'interval': 'month_of_year'},
        )
        histogram_response.raise_for_status()
        monthly = histogram_response.json().get('results', {}).get('month_of_year', {})
        counts_by_month = {int(k): int(v) for k, v in monthly.items()}

        if counts_by_month:
            top3 = sorted(counts_by_month, key=counts_by_month.get, reverse=True)[:3]
            overlap = sc['peak_months'] & set(top3)
            status = 'OK' if overlap else f'Peak months {top3} did not overlap expected {sorted(sc["peak_months"])}'
        else:
            top3 = []
            status = 'No histogram data returned'

        print(
            f"  {sc['common_name']:30s} expect=native "
            f"native_results={native_count} peak_months={top3} {status}"
        )

    print()
    print('If checks fail, review native=true behavior before using outreach outputs.')

Spot-check results:

  Monarch                        expect=native    native_results=1  peak_months=[8, 9, 10]  ✓
  Eastern Tiger Swallowtail      expect=native    native_results=0  peak_months=[1, 2, 3]  ✗ peak months [1, 2, 3] — expected overlap with [5, 6, 8]
  European Honeybee              expect=absent    native_results=0  introduced_results=1  ✓

If any ✗ appears above, re-examine native=true behavior before using outreach charts.


In [11]:
# -- Weekly overview: selected taxa only, styled for presentation --
import plotly.graph_objects as go
from plotly.subplots import make_subplots

selected_taxa = get_active_taxa()
focus_to_taxon = {t['focus_group']: t for t in selected_taxa}

weeks = list(range(1, 53))
month_ticks = [1, 5, 9, 14, 18, 22, 27, 31, 35, 40, 44, 48]
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
life_stage_dash = {'overall': 'solid', 'adult': 'dash', 'larva': 'dot', 'pupa': 'dashdot'}

weekly = hist_df[hist_df['interval'] == 'week_of_year'].copy()
weekly = weekly[weekly['focus_group'].isin(focus_to_taxon.keys())]

if weekly.empty:
    print('No weekly data found for selected taxa.')
else:
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        subplot_titles=(
            'Raw native observations by week',
            'Effort-normalized share of all observations by week',
        ),
        vertical_spacing=0.09,
    )

    for (focus_group, life_stage), group in weekly.groupby(['focus_group', 'life_stage_bucket']):
        taxon = focus_to_taxon[focus_group]
        line = {
            'color': taxon['color'],
            'dash': life_stage_dash.get(life_stage, 'solid'),
            'width': 2.2,
        }
        name = f"{taxon['title_label']} - {life_stage}"

        group = group.set_index('bin').reindex(weeks, fill_value=0)

        fig.add_scatter(
            x=weeks,
            y=group['count'],
            mode='lines',
            name=name,
            line=line,
            legendgroup=name,
            row=1,
            col=1,
        )
        fig.add_scatter(
            x=weeks,
            y=group['fraction'],
            mode='lines',
            name=name,
            line=line,
            legendgroup=name,
            showlegend=False,
            row=2,
            col=1,
        )

    selected_label = ', '.join(t['title_label'] for t in selected_taxa)
    title = f'Weekly native phenology - {selected_label} (VA Piedmont)'
    fig = apply_plot_style(fig, title_text=title, height=760)

    fig.update_xaxes(tickvals=month_ticks, ticktext=month_names)
    fig.update_yaxes(title_text='Observations', row=1, col=1)
    fig.update_yaxes(title_text='Share of all observations', tickformat='.1%', row=2, col=1)
    fig.show()

In [12]:
# -- Top native taxa for selected groups, with taxon-aware thresholding --
selected_taxa = get_active_taxa()

top_rows = []
monthly_rows = []
threshold_summary = []

for taxon in selected_taxa:
    species_counts_response = session.get(
        SPECIES_COUNTS_URL,
        params={
            **_base,
            'taxon_id': taxon['taxon_id'],
            'native': 'true',
            'per_page': TOP_N,
            **taxon.get('extra', {}),
        },
    )
    species_counts_response.raise_for_status()

    group_rows = []
    for rec in species_counts_response.json().get('results', []):
        taxon_info = rec.get('taxon', {})
        group_rows.append(
            {
                'focus_group': taxon['focus_group'],
                'title_label': taxon['title_label'],
                'taxon_id': taxon_info.get('id'),
                'taxon_name': taxon_info.get('name'),
                'common_name': taxon_info.get('preferred_common_name'),
                'total_obs': rec.get('count'),
                'parent_taxon_key': taxon['key'],
            }
        )

    group_df = pd.DataFrame(group_rows)
    if group_df.empty:
        threshold_summary.append((taxon['title_label'], 0, 0, 0))
        continue

    year_response = session.get(
        HISTOGRAM_URL,
        params={
            **_base,
            'taxon_id': taxon['taxon_id'],
            'interval': 'year',
            'native': 'true',
            **taxon.get('extra', {}),
        },
    )
    year_response.raise_for_status()
    yearly_counts = {
        pd.to_datetime(year_key).year: int(year_count)
        for year_key, year_count in year_response.json().get('results', {}).get('year', {}).items()
    }

    active_years = sum(count >= MIN_OBS_PER_YEAR for count in yearly_counts.values())
    active_years = max(active_years, 1)
    min_total_obs = MIN_OBS_PER_YEAR * active_years

    group_df['total_obs'] = pd.to_numeric(group_df['total_obs'], errors='coerce').fillna(0).astype(int)
    group_df = group_df[group_df['total_obs'] >= min_total_obs].copy()

    threshold_summary.append((taxon['title_label'], active_years, min_total_obs, len(group_df)))

    if group_df.empty:
        continue

    top_rows.extend(group_df.to_dict(orient='records'))

    for _, row in group_df.iterrows():
        monthly_response = session.get(
            HISTOGRAM_URL,
            params={
                **_base,
                'taxon_id': int(row['taxon_id']),
                'interval': 'month_of_year',
                **taxon.get('extra', {}),
            },
        )
        monthly_response.raise_for_status()
        for month_key, count in monthly_response.json().get('results', {}).get('month_of_year', {}).items():
            monthly_rows.append(
                {
                    'focus_group': row['focus_group'],
                    'title_label': row['title_label'],
                    'taxon_id': row['taxon_id'],
                    'taxon_name': row['taxon_name'],
                    'common_name': row['common_name'],
                    'month': int(month_key),
                    'count': int(count),
                }
            )

top_taxa_df = pd.DataFrame(top_rows)
monthly_taxa_df = pd.DataFrame(monthly_rows)

print('Threshold summary (selected taxa only):')
for title_label, active_years, min_total_obs, retained_taxa in threshold_summary:
    print(
        f'- {title_label}: active_years={active_years}, '
        f'min_total_obs={min_total_obs}, retained_top_taxa={retained_taxa}'
    )

if not top_taxa_df.empty:
    preview = top_taxa_df[
        ['title_label', 'common_name', 'taxon_name', 'total_obs']
    ].sort_values(['title_label', 'total_obs'], ascending=[True, False])
    print()
    print(preview.to_string(index=False))
else:
    print('\nNo taxa met threshold in this run.')

Threshold summary (selected taxa only):
- Butterflies and Moths: active_years=26, min_total_obs=650, retained_top_taxa=25

          title_label                   common_name           taxon_name  total_obs
Butterflies and Moths     Eastern Tiger Swallowtail      Papilio glaucus       7392
Butterflies and Moths                 Huron Skipper     Atalopedes huron       5545
Butterflies and Moths                       Monarch     Danaus plexippus       5283
Butterflies and Moths                Pearl Crescent     Phyciodes tharos       3479
Butterflies and Moths           Eastern Tailed-Blue      Cupido comyntas       3223
Butterflies and Moths        Silver-spotted Skipper    Epargyreus clarus       2746
Butterflies and Moths           Red-spotted Admiral   Limenitis arthemis       2562
Butterflies and Moths                Common Buckeye       Junonia coenia       2430
Butterflies and Moths Eastern Tent Caterpillar Moth Malacosoma americana       2053
Butterflies and Moths        Ailanthu

In [13]:
# -- Monthly outreach chart: normalized and selection-aware --
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

selected_taxa = get_active_taxa()
focus_to_taxon = {t['focus_group']: t for t in selected_taxa}

if monthly_taxa_df.empty:
    print('No taxa passed threshold for the current selection.')
else:
    monthly_effort = effort_df[
        effort_df['interval'] == 'month_of_year'
    ][['bin', 'effort']].rename(columns={'bin': 'month'})

    monthly_taxa_df = monthly_taxa_df.merge(monthly_effort, on='month', how='left')
    monthly_taxa_df['normalized_count'] = monthly_taxa_df['count'] / monthly_taxa_df['effort'].replace(0, float('nan'))
    monthly_taxa_df['month_name'] = monthly_taxa_df['month'].apply(lambda m: month_order[m - 1])
    monthly_taxa_df['month_name'] = pd.Categorical(monthly_taxa_df['month_name'], categories=month_order, ordered=True)
    monthly_taxa_df['label'] = monthly_taxa_df['common_name'].fillna(monthly_taxa_df['taxon_name'])

    for focus_group, group_df in monthly_taxa_df.groupby('focus_group'):
        taxon = focus_to_taxon.get(focus_group)
        if taxon is None:
            continue

        fig = px.line(
            group_df.sort_values('month'),
            x='month_name',
            y='normalized_count',
            color='label',
            markers=True,
            category_orders={'month_name': month_order},
            labels={
                'month_name': '',
                'normalized_count': 'Share of all observations',
                'label': 'Taxon',
            },
        )

        title = (
            f"Monthly prevalence - top {TOP_N} native {taxon['title_label']} "
            f"(normalized by total observations, threshold {MIN_OBS_PER_YEAR}/year)"
        )
        fig = apply_plot_style(fig, title_text=title, height=620)
        fig.update_yaxes(tickformat='.2%')
        fig.show()

    if len(selected_taxa) > 1:
        compare_df = (
            monthly_taxa_df.groupby(['focus_group', 'month_name'], as_index=False)['normalized_count']
            .sum()
            .sort_values('month_name')
        )
        compare_fig = px.line(
            compare_df,
            x='month_name',
            y='normalized_count',
            color='focus_group',
            markers=True,
            category_orders={'month_name': month_order},
            labels={
                'month_name': '',
                'normalized_count': 'Total share of all observations',
                'focus_group': 'Selected taxon',
            },
            color_discrete_map={fg: focus_to_taxon[fg]['color'] for fg in focus_to_taxon},
        )
        compare_fig = apply_plot_style(compare_fig, title_text='Compare mode: monthly normalized prevalence by selected taxon', height=560)
        compare_fig.update_yaxes(tickformat='.2%')
        compare_fig.show()

In [14]:
# -- Life stage annotation coverage report (only for selected taxa with stage definitions) --
selected_taxa = get_active_taxa()
stage_capable_taxa = [t for t in selected_taxa if t.get('life_stages')]

if not stage_capable_taxa:
    print('No selected taxa define life-stage slices; skipping coverage report.')
else:
    print('Life stage annotation coverage:')
    for taxon in stage_capable_taxa:
        focus_group = taxon['focus_group']
        overall = hist_df[
            (hist_df['focus_group'] == focus_group)
            & (hist_df['life_stage_bucket'] == 'overall')
            & (hist_df['interval'] == 'week_of_year')
        ]['count'].sum()

        print()
        print(f"{taxon['title_label']} - native observations (all-time): {int(overall):,}")
        for stage in taxon['life_stages']:
            stage_count = hist_df[
                (hist_df['focus_group'] == focus_group)
                & (hist_df['life_stage_bucket'] == stage['bucket'])
                & (hist_df['interval'] == 'week_of_year')
            ]['count'].sum()
            pct = (stage_count / overall) if overall else float('nan')
            caution = ' (sparse, interpret cautiously)' if pct < 0.10 else ''
            print(f"  - {stage['bucket']}: {int(stage_count):,} ({pct:.1%}){caution}")

Life stage annotation coverage:

Butterflies and Moths - native observations (all-time): 164,298
  - adult: 110,085 (67.0%)
  - larva: 22,626 (13.8%)
  - pupa: 749 (0.5%) (sparse, interpret cautiously)
